In [ ]:
import os
import re
import json
import random
import pickle
import numpy as np
import pandas as pd
from rich.pretty import pprint
import matplotlib.pyplot as plt
from collections import defaultdict
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction
from pathlib import Path
import json

path_data = Path('/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/restricted/anonymization/train.txt')
#path_data = Path('/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/annotations.conll')
output_path = Path('/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/annotations_entities.json')


## Convert CoNLL annotations to JSON entities

In [ ]:
current_label = None
sort_alphabetically = False

entities = []
current_tokens = []

with path_data.open(encoding='utf-8') as handle:
    for raw_line in handle:
        line = raw_line.strip()

        if not line or raw_line.startswith('-DOCSTART-'):
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None
            continue

        parts = raw_line.split()
        token, tag = parts[0], parts[-1]

        if tag == 'O':
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None
            continue

        if '-' not in tag:
            continue

        prefix, label = tag.split('-', 1)

        if prefix == 'B':
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = [token]
            current_label = label
        elif prefix == 'I' and current_label == label:
            current_tokens.append(token)
        else:
            if current_label and current_tokens:
                entities.append({current_label: ' '.join(current_tokens)})
            current_tokens = []
            current_label = None

if current_label and current_tokens:
    entities.append({current_label: ' '.join(current_tokens)})

if sort_alphabetically:
    # Convert dicts to tuples for sorting
    ordered_entities = sorted(
        entities,
        key=lambda item: (list(item.keys())[0], list(item.values())[0])
    )
    entities = ordered_entities

output_path.write_text(json.dumps({'extractions': entities}, ensure_ascii=False, indent=2) + '', encoding='utf-8')
print(f'Wrote {len(entities)} entities to {output_path}')

entities[:10]


## Langextract functions